In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

# Detect data root (Kaggle or local)
possible_roots = [
    Path("/kaggle/input"),
    Path("./data"),
    Path("./"),
]
DATA_ROOT = None
for root in possible_roots:
    if (root / "hms-harmful-brain-activity-classification").exists():
        DATA_ROOT = root / "hms-harmful-brain-activity-classification"
        break
if DATA_ROOT is None:
    DATA_ROOT = Path(".")  # fallback


def load_csv(fname: str) -> pd.DataFrame:
    """Load a CSV searching common locations."""
    candidates = [
        Path(fname),
        DATA_ROOT / fname,
    ]
    for p in candidates:
        if p.is_file():
            return pd.read_csv(p)
    raise FileNotFoundError(f"Could not find {fname} in any searched location.")


# ------------------------------------------------------------------
# Load training data and compute row‑wise probability vectors
df = load_csv("train.csv")
TARGETS = df.columns[-6:]  # the six vote columns
print("Train shape:", df.shape)
print("Targets:", list(TARGETS))

row_sums = df[TARGETS].sum(axis=1).replace(0, np.nan)
row_probs = df[TARGETS].div(row_sums, axis=0).fillna(0)

# Weight each row by the total number of votes (annotators)
vote_weights = df[TARGETS].sum(axis=1).fillna(0).values
df["vote_weights"] = vote_weights

# Global marginal (vote‑weighted)
marginal_probs = np.average(row_probs.values, axis=0, weights=vote_weights)
marginal_probs = marginal_probs / marginal_probs.sum()
print("Baseline global marginal (weighted):", marginal_probs)

# Patient‑level marginals
patient_weighted_sum = (
    row_probs.multiply(vote_weights, axis=0).groupby(df["patient_id"]).sum()
)
patient_weights = df.groupby("patient_id")["vote_weights"].sum()
patient_marginals = patient_weighted_sum.div(patient_weights, axis=0)
patient_marginals = patient_marginals.div(patient_marginals.sum(axis=1), axis=0)

# EEG‑level marginals
eeg_weighted_sum = row_probs.multiply(vote_weights, axis=0).groupby(df["eeg_id"]).sum()
eeg_weights = df.groupby("eeg_id")["vote_weights"].sum()
eeg_marginals = eeg_weighted_sum.div(eeg_weights, axis=0)
eeg_marginals = eeg_marginals.div(eeg_marginals.sum(axis=1), axis=0)




Train shape: (96950, 15)
Targets: ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
Baseline global marginal (weighted): [0.12055157 0.1575504  0.17107176 0.13311643 0.14810585 0.26960399]


In [2]:
# Load test metadata
test = load_csv("test.csv")
print("Test shape:", test.shape)

n_test = test.shape[0]
preds = np.empty((n_test, len(TARGETS)), dtype=np.float32)

# Updated blending weights – more reliance on specific priors
w_eeg = 0.95  # EEG‑level when present
w_patient_when_eeg = 0.04  # extra patient weight when EEG is present
w_patient_alone = 0.97  # patient‑level when EEG missing
epsilon = 1e-6  # slightly larger smoothing

for i, (_, row) in enumerate(test.iterrows()):
    eid = row["eeg_id"]
    pid = row["patient_id"]
    if eid in eeg_marginals.index:
        eeg_vec = eeg_marginals.loc[eid].values
        patient_vec = (
            patient_marginals.loc[pid].values
            if pid in patient_marginals.index
            else marginal_probs
        )
        pred = (
            w_eeg * eeg_vec
            + w_patient_when_eeg * patient_vec
            + (1.0 - w_eeg - w_patient_when_eeg) * marginal_probs
        )
    elif pid in patient_marginals.index:
        pred = (
            w_patient_alone * patient_marginals.loc[pid].values
            + (1.0 - w_patient_alone) * marginal_probs
        )
    else:
        pred = marginal_probs.copy()
    # smoothing and renormalisation
    pred = pred + epsilon
    pred = pred / pred.sum()
    preds[i] = pred

# Build submission
sub = pd.DataFrame({"eeg_id": test["eeg_id"].values})
sub[TARGETS] = preds

sub_path = "submission.csv"
sub.to_csv(sub_path, index=False)
print("Submission saved to", sub_path)
print("Submission shape:", sub.shape)
print("Row sums (first 5, should be ~1.0):")
print(sub[TARGETS].sum(axis=1).head())

Test shape: (9850, 3)
Submission saved to submission.csv
Submission shape: (9850, 7)
Row sums (first 5, should be ~1.0):
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
dtype: float32
